# Hybrid Query test and benchmark design

This specification extends [Core and native architecture](core-native-architecture.ipynb). Hybrid Query means DuckDB SQL composed with `GRAPH_TABLE`. It covers reproducible SQLLogic checks, matched performance comparison, and an evidence trail from a benchmark series back to the SQLLogic case, fixture, extension artifact and runtime boundary.

The design reuses three entry points. The stabilization runner tests the public SQL surface, the provider runner tests dispatch and graph-fixture immutability, and the provider tracker runs measured workers and publishes verified snapshots. [Stabilization runner](../test/stabilization/run_sqllogictest.py) · [Provider runner](../test/relate/run.py) · [Benchmark tracker](../benchmarks/provider_comparison/tracker.py)

The two architecture views show where exact outcomes are checked and where performance evidence is captured. In the existing execution path, DuckDB owns the residual join and final result. Core owns binding, admission and route selection. Native adapters own provider sessions and row delivery. The evidence pipeline keeps SQL result rows, graph scan output rows and provider-reported timings at their respective boundaries.

In [ ]:
architecture-beta
    %% @ns-zone-allow test runtime
    %% @ns-zone-allow data runtime
    %% @ns-zone-allow data test
    group test(cloud)[Correctness runners]
    group runtime(cloud)[Native execution]
    group data(cloud)[Frozen workload]
    service slt(server)[DuckDB SQLLogic] in test
    service oracle(database)[Relational SQL Oracle] in data
    service fixture(database)[Graph and SQL Fixtures] in data
    service duck(server)[DuckDB SQL and Optimizer] in runtime
    service core(server)[Core Bind and Scan] in runtime
    service adapter(server)[Native Provider Adapter] in runtime
    service graphdb(database)[Read Only Graph] in runtime
    slt:R --> L:duck
    fixture:T --> B:duck
    fixture:R --> L:graphdb
    fixture:L --> R:oracle
    oracle:T --> B:slt
    duck:R --> L:core
    core:R --> L:adapter
    adapter:R --> L:graphdb

In [ ]:
architecture
    %% @ns-zone-allow worker gates
    group worker(cloud)[Pinned benchmark host]
    group gates(cloud)[Evidence and publication]
    service release(server)[Release worker] in worker
    service trace(server)[Diagnostic worker] in worker
    service tracker(server)[Provider tracker] in worker
    service slt(server)[SQLLogic gates] in gates
    service review(server)[Paired advisory] in gates
    service history(database)[Immutable attempt history] in gates
    tracker:R --> L:release
    tracker:B --> T:trace
    release:R --> L:slt
    trace:R --> L:slt
    slt:R --> L:review
    review:R --> L:history

## Workload definitions and SQLLogic contracts

Keep the existing `# case:` markers and [stabilization coverage manifest](../test/stabilization/case_manifest.csv). Add a versioned workload manifest that references those markers and the tracker case registry. A single logical case can have a hand-sized fixture and a scaled instance of the same query template. Closely related queries with different semantics get their own cases. Define typed parameter vectors and their order once, then use the same vectors for both arms of every comparable instance.

| Record | Required identity and fields |
|---|---|
| Case | `(case_id, case_revision)`, SQL template digest, output schema, bag or ordered mode, oracle ID/revision, route and provider eligibility, SQLLogic references |
| Instance | `(case_id, case_revision, scale, fixture_id, parameter_trace_id)`, logical input SHA-256, ordered parameter bytes, generator and format revisions, reference result rows or digest |
| Run | `run_id`, instance reference, tracker protocol and schedule digest, thread/session controls, provider/client versions, DuckDB build, build source, extension and fixture materialization SHA-256, provenance for release and any diagnostic binary |
| Receipt | `(run_id, case_id, provider, arm, block, repetition)` for calls, or a typed gate receipt with SQLLogic file and source hash, record count, pass/fail status and failure reason |
| Observation | Receipt or run reference, metric ID, value and unit, sample grain, phase/capture, and operator or scan identity when there can be multiple scans |

Use an independent relational oracle over the fixture's vertex, edge and attribute tables. Small cases assert literal typed rows, with explicit schema checks for DuckDB SQLLogic. Unordered results compare as bags using `EXCEPT ALL` in both directions or an equivalent whole-row comparison. Ordered queries use an observable order and stable tie-breakers where the contract permits. Keep the current ordered-prefix positive case at its supported single visible order value. Assert fallback for variants with residual filters, offsets, unsupported order and richer projections. For scaled exact outputs, store the encoding version, schema, row count and SHA-256 of the canonical typed rows. Keep the incumbent CSV-based oracle and integer signatures for current compatible workloads, and introduce richer encodings when new cases require them.

| Existing suite | Contract to reuse |
|---|---|
| `hybrid_public.test` and `usage_binding.test` | Public SQL composition, keys and NULLs, result bags, join/LIMIT placement and registration lifecycle |
| `watchlist_applied_route.test`, `watchlist_offer_diagnostics.test`, `ordered_watchlist_limit.test` | Prepared execution, offer eligibility versus applied route, fallback, provider capability, graph rows crossing the scan boundary |
| `micro_results.test` and `micro_snapshot.test` | Scaled exact results and snapshot gates. The tracker already checks each full fetch against its oracle and runs these checks before publishing a MICRO snapshot. |

For local correctness use isolated fixture state and the pinned DuckDB SQLLogic runner. For controlled Neo4j keep provider integration cases independent of measured local arms. On every PR cover essential public semantics and high-risk route boundaries. Grow scheduled coverage with a constrained matrix over join shape, key cardinality, route settings, provider and fixture scale. Explicitly include cap-minus-one/cap/cap-plus-one, a stale generation, key duplicates, missing/NULL keys, parallel edges, empty results, and wrong LIMIT placement.

[Tiny hybrid tests](../test/stabilization/sqllogic/hybrid_public.test) · [Usage binding](../test/stabilization/sqllogic/usage_binding.test) · [Applied route tests](../test/relate/watchlist_applied_route.test) · [Offer diagnostics](../test/relate/watchlist_offer_diagnostics.test) · [Ordered watchlist](../test/relate/ordered_watchlist_limit.test) · [Scaled result gate](../benchmarks/provider_comparison/tracker/micro_results.test) · [Snapshot gate](../benchmarks/provider_comparison/tracker/micro_snapshot.test)

## Execution and evidence boundaries

The sequence follows one successful release submission and a later diagnostic query. Start release wall and CPU clocks at submission and stop them when the Arrow table is fully fetched. Compare the schema and result after stopping the timer. Fetch, validate and retain each sample independently. A timeout, wrong result or invalid artifact causes a failed attempt before snapshot publication. The first execution in each worker/case is the tracker warmup. Report it as first fetch on a fresh session with the filesystem cache policy recorded. Warm percentiles come from subsequent verified calls.

A dispatch diagnostic records `QuackWeaveWatchlist` offer/decline, `QuackWeaveGraphAdmission` generation, and `QuackWeaveGraphRoute` actual dispatch. Preserve provider route enums `host`, `literal_exact`, `runtime_exact` and `bounded_exact` where the core emits them. Tracker arms such as `qw_pgq` and `qw_pgq_host` describe settings, so store the applied route separately. A prepared query can bind one generation and fail admission after replacement. Route emission follows reader dispatch. Completed scan feedback applies at verified EOF for eligible scans. The state diagrams encode verified sample retention and admission for completed diagnostics.

[Route decision](../crates/core/src/qw_query/projection.rs) · [Core dispatch and route logging](../crates/core/src/qw_query/runtime_watch.rs) · [Native optimizer offer](../crates/native/grammar/runtime_watchlist.cpp) · [Ordered-prefix eligibility](../crates/native/grammar/ordered_watchlist.hpp)

In [ ]:
sequenceDiagram
    participant B as Tracker
    participant D as DuckDB
    participant C as Core scan
    participant P as Native provider
    participant O as Oracle
    participant A as Snapshot gate
    Note over B,A: @spec VERIFIED_SAMPLE_FLOW
    B->>D: Start clock and submit SQL
    Note over B,D: @message SUBMIT<br/>@from B<br/>@to D<br/>@event Start clock and submit SQL<br/>@order 1<br/>@when true<br/>@ensures SUBMIT_OK: true
    D->>C: Initialize graph scan and offer keys
    Note over D,C: @message INIT<br/>@from D<br/>@to C<br/>@event Initialize graph scan and offer keys<br/>@order 2<br/>@when true<br/>@ensures INIT_OK: true
    C->>P: Admit generation and dispatch
    Note over C,P: @message OPEN<br/>@from C<br/>@to P<br/>@event Admit generation and dispatch<br/>@order 3<br/>@when true<br/>@ensures OPEN_OK: true
    P->>C: Return typed batch stream
    Note over P,C: @message ROWS<br/>@from P<br/>@to C<br/>@event Return typed batch stream<br/>@order 4<br/>@when true<br/>@ensures ROWS_OK: true
    C->>D: Deliver graph rows to residual SQL
    Note over C,D: @message CHUNK<br/>@from C<br/>@to D<br/>@event Deliver graph rows to residual SQL<br/>@order 5<br/>@when true<br/>@ensures CHUNK_OK: true
    D->>B: Finish Arrow fetch and stop clock
    Note over D,B: @message FETCH<br/>@from D<br/>@to B<br/>@event Finish Arrow fetch and stop clock<br/>@order 6<br/>@when true<br/>@ensures FETCH_OK: true
    B->>O: Compare typed result with oracle
    Note over B,O: @message VERIFY<br/>@from B<br/>@to O<br/>@event Compare typed result with oracle<br/>@order 7<br/>@when true<br/>@ensures VERIFY_OK: true
    B->>D: Run untimed route and row probe
    Note over B,D: @message PROBE<br/>@from B<br/>@to D<br/>@event Run untimed route and row probe<br/>@order 8<br/>@when true<br/>@ensures PROBE_OK: true
    B->>A: Submit candidate snapshot to SQLLogic
    Note over B,A: @message SAVE<br/>@from B<br/>@to A<br/>@event Submit candidate snapshot to SQLLogic<br/>@order 9<br/>@when true<br/>@ensures SAVE_OK: true
    Note over B,A: @verify TRACE: prove sequence_protocol

In [ ]:
stateDiagram-v2
    [*] --> Prepared
    Prepared --> Fetched: fetch
    Prepared --> Failed: execution_error
    Fetched --> Valid: verify
    Fetched --> Failed: mismatch
    note right of Prepared
      @spec SAMPLE_EVIDENCE
      @type Event = enum[fetch,execution_error,verify,mismatch]
      @input event: Event
      @state-var fetched: Bool
      @state-var oracle_ok: Bool
      @state-var retained: Bool
      @requires PRE: fetched = false and oracle_ok = false and retained = false
      @state Prepared
      @invariant KEEP_VERIFIED_ONLY: retained = false or (fetched = true and oracle_ok = true)
    end note
    note right of Fetched
      @state Fetched
    end note
    note right of Failed
      @state Failed
    end note
    note right of Valid
      @state Valid
    end note
    note right of Fetched
      @transition T_fetch
      @from Prepared
      @to Fetched
      @event event = fetch
      @guard retained = false
      @update fetched' = true
      @update oracle_ok' = false
      @update retained' = false
    end note
    note right of Failed
      @transition T_execution_error
      @from Prepared
      @to Failed
      @event event = execution_error
      @guard retained = false
      @update fetched' = false
      @update oracle_ok' = false
      @update retained' = false
    end note
    note right of Valid
      @transition T_verify
      @from Fetched
      @to Valid
      @event event = verify
      @guard fetched = true and retained = false
      @update oracle_ok' = true
      @update retained' = true
    end note
    note right of Failed
      @transition T_mismatch
      @from Fetched
      @to Failed
      @event event = mismatch
      @guard fetched = true and retained = false
      @update oracle_ok' = false
      @update retained' = false
    end note
    note right of Prepared
      @verify INIT: prove initiate KEEP_VERIFIED_ONLY
      @verify ON_fetch: prove preserve KEEP_VERIFIED_ONLY on T_fetch
      @verify ON_execution_error: prove preserve KEEP_VERIFIED_ONLY on T_execution_error
      @verify ON_verify: prove preserve KEEP_VERIFIED_ONLY on T_verify
      @verify ON_mismatch: prove preserve KEEP_VERIFIED_ONLY on T_mismatch
    end note

In [ ]:
stateDiagram
    [*] --> Bound
    Bound --> Admitted: open_current
    Bound --> Rejected: reject_stale
    Admitted --> Complete: verified_eof
    Admitted --> Rejected: read_error
    note right of Bound
      @spec GENERATION_DIAGNOSTIC_EVIDENCE
      @type Event = enum[open_current,reject_stale,verified_eof,read_error]
      @input event: Event
      @state-var admitted: Bool
      @state-var reported: Bool
      @requires PRE: admitted = false and reported = false
      @state Bound
      @invariant REPORTED_REQUIRES_ADMISSION: reported = false or admitted = true
    end note
    note right of Admitted
      @state Admitted
    end note
    note right of Rejected
      @state Rejected
    end note
    note right of Complete
      @state Complete
    end note
    note right of Admitted
      @transition T_open_current
      @from Bound
      @to Admitted
      @event event = open_current
      @guard admitted = false and reported = false
      @update admitted' = true
    end note
    note right of Rejected
      @transition T_reject_stale
      @from Bound
      @to Rejected
      @event event = reject_stale
      @guard admitted = false and reported = false
      @update admitted' = false
      @update reported' = false
    end note
    note right of Complete
      @transition T_verified_eof
      @from Admitted
      @to Complete
      @event event = verified_eof
      @guard admitted = true and reported = false
      @update reported' = true
    end note
    note right of Rejected
      @transition T_read_error
      @from Admitted
      @to Rejected
      @event event = read_error
      @guard admitted = true and reported = false
      @update reported' = false
    end note
    note right of Bound
      @verify INIT: prove initiate REPORTED_REQUIRES_ADMISSION
      @verify ON_open_current: prove preserve REPORTED_REQUIRES_ADMISSION on T_open_current
      @verify ON_reject_stale: prove preserve REPORTED_REQUIRES_ADMISSION on T_reject_stale
      @verify ON_verified_eof: prove preserve REPORTED_REQUIRES_ADMISSION on T_verified_eof
      @verify ON_read_error: prove preserve REPORTED_REQUIRES_ADMISSION on T_read_error
    end note

## Metric and history contract

Use the following registry fields for observation validation. "Capture" describes where the observation was obtained. Add stable metric IDs to a normalized history view next to the current summary and bootstrap macros. Keep raw samples and probe records addressable by run, arm, case and observation grain.

| Metric ID | Architecture stage | Unit | Capture | Grain | Evidence in the existing flow or proposed projection |
|---|---|---|---|---|---|
| `release_wall` | client | ms | release | query | `workers[].samples[].wall_ms` · matched full-fetch p50/p95 |
| `release_cpu` | client | ms | release | query | `workers[].samples[].cpu_ms` · process CPU from the same call |
| `bootstrap` | host_setup | ms | release | connection | `workers[].setup.bootstrap_ms` · open/load/register/select/session setup per connection |
| `first_fetch` | client | ms | release | connection_case | `samples[repetition=0].wall_ms` · expose separately per case/worker |
| `engine_exec` | provider | ms | diagnostic | query | `engine_samples[].engine_ms` · provider-native timer on verified diagnostics |
| `engine_remainder` | client | ms | diagnostic | query | `other_warm_work` · same diagnostic call elapsed minus its own engine interval |
| `scan_rows` | scan_boundary | rows | explain | query_scan | `workers[].graph_scan_rows` · separate `EXPLAIN ANALYZE` scan operator cardinality |
| `result_rows` | sql_output | rows | oracle | query | `expected[case][0]` · exact oracle and actual result count |
| `result_digest` | sql_output | sha256 | oracle | query | `expected[case][1]` · compare each fetched result to the oracle |
| `applied_route` | scan_init | enum | logs | query_scan | Core `QuackWeaveGraphRoute` diagnostic · retain with scan identity |
| `admitted_generation` | scan_admission | opaque | logs | query_scan | Core `QuackWeaveGraphAdmission` diagnostic · keep local to the registration instance |
| `peak_rss` | worker | mib | release | process | `workers[].peak_rss_mib` · worker high-water mark |

Use millisecond doubles in tracker JSON and report p50 as median and p95 as nearest-rank with the verified sample count. Use integer rows for scan and result counts and preserve multiplicities. The scan count measures graph rows delivered to DuckDB before residual SQL. The final result count measures the SQL result. Worker RSS is per process and bootstrap is per connection. When the companion lacks a timer for an arm, use a null observation with an availability reason. The tracker currently collects hybrid graph scan rows through a separate plan probe and provider engine samples on eligible standalone routes. Additional startup/first-fetch views, route/generation records, per-scan dimensions, and richer metric grains are extensions of the current JSON.

For comparison, extend `ci.py cohort()` and `verify_pair()` with workload/template/oracle/parameter and schedule identity when those fields are introduced. Require equal fixture and tracker hashes, host pairing, host and client digests, protocol and series keys. Match baseline and candidate artifact digests to their own release manifests. Alternate block order and compare p50/p95 by provider, arm and case instance. Keep the existing 10% warning for either eligible percentile advisory. New protocols and previously unseen cases become fresh baselines. Failed correctness or evidence checks reject publication; failed or unavailable attempts retain status and reason in the current CI evidence path.

The queryable lineage is `history series → run/attempt → workload instance → case revision → SQLLogic evidence`. Attach transport or diagnostic data by its own receipt and scan ID. Keep `history.sql` additive for old snapshots and derive report tables in the current renderer/export path. Bump the benchmark protocol when an interval, unit, materialization or comparison changes the meaning of a metric.

[Worker timing](../benchmarks/provider_comparison/tracker.py) · [Provider diagnostic collection](../benchmarks/provider_comparison/engine_timing.py) · [History queries](../benchmarks/provider_comparison/tracker/history.sql) · [Pairing and advisory](../benchmarks/provider_comparison/tracker/ci.py) · [CI workflow](../.github/workflows/Benchmark.yml)

## Correctness and comparison gates

Before timing, verify release identity, DuckDB version and immutable fixture input. Use the existing CI correctness job for public/provider SQLLogic and the existing benchmark outcome checks in the tracker. Reuse the oracle comparison after every measured or warmup query. Before publishing a snapshot, require expected SQLLogic outcomes, complete expected arm/case/worker series, consistent evidence hashes, and successful fixture readback. Include explicit status for an unsupported arm, a correctness failure, and unavailable diagnostics.

The flowchart proves that the modeled publication branches are exclusive, exhaustive and deterministic. The graph classifies a verified attempt as baseline, paired, or incompatible under the declared compatibility inputs. An incompatible historical attempt can be kept for traceability and considered under a fresh compatible series policy. The current CI comparison independently checks baseline artifact provenance, so a wrong restored artifact still fails pairing.

In [ ]:
flowchart TD
    S["`@spec VERIFIED_SNAPSHOT_PUBLICATION
@type Status = enum[publishable,failed]
@input artifacts_ok: Bool
@input fixtures_ok: Bool
@input oracle_ok: Bool
@input snapshot_ok: Bool
@output status: Status
@requires PRE: true`"]
    B0["`@branch PUBLISH
@when artifacts_ok = true and fixtures_ok = true and oracle_ok = true and snapshot_ok = true
@ensures PUBLISH_RESULT: status = publishable`"]
    B1["`@branch REJECT
@when artifacts_ok = false or fixtures_ok = false or oracle_ok = false or snapshot_ok = false
@ensures REJECT_RESULT: status = failed`"]
    V["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify DISJOINT: prove partition_exclusive
@verify OUTCOMES: witness each status`"]
    S --> B0 --> V
    S --> B1 --> V

In [ ]:
graph TD
    S["`@spec BASELINE_CLASSIFICATION
@type Status = enum[baseline,paired,incompatible]
@input has_baseline: Bool
@input same_workload: Bool
@input same_protocol: Bool
@input same_host: Bool
@input artifact_matches: Bool
@output status: Status
@requires PRE: true`"]
    B0["`@branch FIRST
@when has_baseline = false
@ensures FIRST_RESULT: status = baseline`"]
    B1["`@branch COMPARE
@when has_baseline = true and same_workload = true and same_protocol = true and same_host = true and artifact_matches = true
@ensures COMPARE_RESULT: status = paired`"]
    B2["`@branch DECLINE
@when has_baseline = true and (same_workload = false or same_protocol = false or same_host = false or artifact_matches = false)
@ensures DECLINE_RESULT: status = incompatible`"]
    V["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify DISJOINT: prove partition_exclusive
@verify OUTCOMES: witness each status`"]
    S --> B0 --> V
    S --> B1 --> V
    S --> B2 --> V

## Example evidence from two local replays

These are `micro_creator_selective` smoke runs on the LSQB SF1 `mod32` creator projection, captured with the prebuilt local extension identified below. Each run used two alternating blocks, two providers and two arms. It verified eight warmups and 16 measured queries, plus exact result and snapshot SQLLogic gates. Both snapshots matched their statement, schedule, policy, fixture and binary identity and returned the same four-row result signature. Values here are fixed projections of the snapshots. Run the Arrow input cell before rerunning the four chart proofs. Each chart binds the matching port, row-content hash and schema hash.

| Capture | Replay 1 | Replay 2 |
|---|---|---|
| Tracker snapshot ID | `20261127T041430Z-44c2653a` | `20261127T041435Z-5a71d51e` |
| Snapshot SHA-256 | `7cd516bd623dd64811a1a6941f726d3bca081b53bb3a03c78050f3c93d7badca` | `39908042997ff183b3074f368ca77c45015559f0813e85698e74efe729835d52` |
| Expected result fingerprint | `7f2df2fb87fce9818960fee6a121a32ab86c7977de5536c33615fbb9b617fe04` | Same |

Fixture CSV SHA-256 `86ee28c86fbc633d3f354721c7778ff404ccf41370d9a74b567717a78c2e9e0b`. Extension SHA-256 `c662638b1fb0dd8459989c916a8f99e6ae172604761f032e16ecd4af75dba176`.

| Provider and arm | Replay 1 p50 / p95 (ms) | Replay 2 p50 / p95 (ms) | Scan rows (both) |
|---|---:|---:|---:|
| Ladybug `qw_pgq` | 2.209 / 2.393 | 2.114 / 2.264 | 734 |
| Ladybug `qw_pgq_host` | 19.469 / 19.911 | 20.039 / 20.634 | 119,052 |
| RelateDB `qw_pgq` | 1.901 / 2.011 | 1.916 / 1.953 | 734 |
| RelateDB `qw_pgq_host` | 116.933 / 119.053 | 113.999 / 116.889 | 119,052 |

The Sankey carries Ladybug's graph scan counts into the diagnostic record in two separate executions. The pie counts verified calls in one replay. The bar charts use integer microseconds for the release median and exact row counts for the scan boundary. Four measurements per series validate replay plumbing. Performance advisories use the normal matched sampling protocol.

[Recorded replay context](core-native-architecture.ipynb) · [Tracker run instructions](../benchmarks/provider_comparison/tracker/README.md#run)

In [2]:
import pyarrow as pa

# Exact integer projections of the tracker snapshots identified above.
# Values are decimal text in the NS-Mermaid Arrow binding.
datasets = [
    {
        "n": "bench_scan_lineage",
        "cols": [
            "source",
            "target",
            "value"
        ],
        "rows": [
            [
                "Ladybug prefilter arm",
                "Prefilter scan endpoint",
                "734"
            ],
            [
                "Prefilter scan endpoint",
                "Prefilter diagnostic record",
                "734"
            ],
            [
                "Ladybug host arm",
                "Host scan endpoint",
                "119052"
            ],
            [
                "Host scan endpoint",
                "Host diagnostic record",
                "119052"
            ]
        ]
    },
    {
        "n": "bench_call_counts",
        "cols": [
            "label",
            "value"
        ],
        "rows": [
            [
                "Warmup calls",
                "8"
            ],
            [
                "Measured calls",
                "16"
            ]
        ]
    },
    {
        "n": "bench_release_p50_us",
        "cols": [
            "category",
            "value"
        ],
        "rows": [
            [
                "Ladybug opt",
                "2209"
            ],
            [
                "Ladybug host",
                "19469"
            ],
            [
                "RelateDB opt",
                "1901"
            ],
            [
                "RelateDB host",
                "116933"
            ]
        ]
    },
    {
        "n": "bench_scan_rows",
        "cols": [
            "category",
            "value"
        ],
        "rows": [
            [
                "Ladybug opt",
                "734"
            ],
            [
                "Ladybug host",
                "119052"
            ],
            [
                "RelateDB opt",
                "734"
            ],
            [
                "RelateDB host",
                "119052"
            ]
        ]
    }
]

for dataset in datasets:
    schema = pa.schema([
        pa.field(name, pa.string(), nullable=False,
                 metadata={b"unit": b"1"} if name == "value" else None)
        for name in dataset["cols"]
    ])
    columns = [pa.array(values, type=pa.string()) for values in zip(*dataset["rows"])]
    spur.put(dataset["n"], pa.Table.from_arrays(columns, schema=schema))


source,target,value
Ladybug prefilter arm,Prefilter scan endpoint,734
Prefilter scan endpoint,Prefilter diagnostic record,734
Ladybug host arm,Host scan endpoint,119052
Host scan endpoint,Host diagnostic record,119052


label,value
Warmup calls,8
Measured calls,16


category,value
Ladybug opt,2209
Ladybug host,19469
RelateDB opt,1901
RelateDB host,116933


category,value
Ladybug opt,734
Ladybug host,119052
RelateDB opt,734
RelateDB host,119052


In [ ]:
sankey
    %% @ns-bind producer=2063cc8a-80d5-40df-a4a4-000000000026 port=bench_scan_lineage snapshot=e83f9deb43b5f80a21f4352726633ba1bdef5f5c93607386e6ed1558a077c2cb schema=44f729ca7d2a5dc225e92c5d599811db8260be3ae36f104f056c5f54d935e768 unit=1
    Ladybug prefilter arm,Prefilter scan endpoint,734
    Prefilter scan endpoint,Prefilter diagnostic record,734
    Ladybug host arm,Host scan endpoint,119052
    Host scan endpoint,Host diagnostic record,119052

In [ ]:
pie title Verified calls in one smoke replay
    %% @ns-bind producer=2063cc8a-80d5-40df-a4a4-000000000026 port=bench_call_counts snapshot=1cc5b24f22f97a0741eb055ef76eb5c664f5cb28377f82990a3638d961b2e696 schema=8ee936a11a67759da60dd14c7f971b7ed31831f78b26bbf77ef932016a3f4edd unit=1
    "Warmup calls" : 8
    "Measured calls" : 16
    %% @ns-total 24

In [ ]:
xychart
    %% @ns-bind producer=2063cc8a-80d5-40df-a4a4-000000000026 port=bench_release_p50_us snapshot=0c20fd66d93edfeaea7cb5e8da11913f2cc26fa35dd7239ab899d7901a7d7873 schema=f7188d1799d47517ce745717ab86a9aa97dbf6c7375e3bca1198b649a90543a8 unit=1
    title "Release p50 in replay 1 (microseconds)"
    x-axis ["Ladybug opt", "Ladybug host", "RelateDB opt", "RelateDB host"]
    y-axis "Microseconds"
    bar [2209, 19469, 1901, 116933]

In [ ]:
xychart-beta
    %% @ns-bind producer=2063cc8a-80d5-40df-a4a4-000000000026 port=bench_scan_rows snapshot=ce1461f3423a0535efc1fd7e7cf3720685e6e06defdc6678b4f0685f9c37eb77 schema=f7188d1799d47517ce745717ab86a9aa97dbf6c7375e3bca1198b649a90543a8 unit=1
    title "Graph rows delivered in both replays"
    x-axis ["Ladybug opt", "Ladybug host", "RelateDB opt", "RelateDB host"]
    y-axis "Graph scan rows"
    bar [734, 119052, 734, 119052]

## Integration order and acceptance

1. Give `micro_creator_selective` the first shared case record. Reference its current tracker query and MICRO SQLLogic gates. Add a tiny fixture for that same template and link it to the public correctness suite. Use distinct records for airport and ordered-prefix cases.
2. Make the two runners and tracker resolve stable case IDs, artifact hashes, and fixture/oracle revisions through their existing entry points. Resolve the same SQL and parameter list from the shared record. Preserve old benchmark reads through additive history fields and explicit protocol cohort rules.
3. Add the metric projection and pairing preflight. Use the existing CI build/restore, measurement and archive steps. Extend the current tracker checks for fixture/SQL/oracle or schedule drift, wrong-release receipts, warning-only performance regressions and absent optional diagnostics.
4. Require two fresh scratch executions of the tiny witness and two scaled smoke replays on the same matched host. Compare result/schema, diagnostic route and row counts where declared, schedules and immutable digests. Inject one lost duplicate, one unsafe prefix/fallback and one mismatched case revision/scale. Each defect must fail the corresponding correctness or identity gate. Verify that benchmark timings remain advisory for compatible correct runs.

The Gantt and timeline solve dependency feasibility with one logical slot per task. Their dates anchor those slots, and their durations express dependencies for the design review.

```sh
make native-stabilization-test-prebuilt
python3 -B test/relate/run.py --duckdb-unittest "$DUCKDB_UNITTEST" \
  --fixture-builder "$QW_RELATE_FIXTURE" --extension "$QW_EXT" --lbug "$LBUG" \
  --case watchlist_applied_route.test --case ordered_watchlist_limit.test
"$TRACKER_PYTHON" -B benchmarks/provider_comparison/tracker.py run \
  --micro --case micro_creator_selective --fixture build/perf/provider-comparison/mod32 \
  --duckdb-unittest "$DUCKDB_UNITTEST" --library "$DUCKDB_LIB" --extension "$QW_EXT" \
  --blocks 2 --runs 2 --out "$NEW_SNAPSHOT"
```

[Provider and stabilization targets](../Makefile) · [CI checks](../.github/workflows/CI.yml) · [Benchmark workflow](../.github/workflows/Benchmark.yml) · [Artifact verification](../benchmarks/provider_comparison/tracker/artifacts.py)

In [ ]:
gantt
    title Dependency order in logical slots
    dateFormat YYYY-MM-DD
    axisFormat %d
    section Workload
    Verify package and fixture :package, 2000-01-01, 1d
    Run exact SQLLogic checks :slt, after package, 1d
    section Benchmark
    Run matched sample blocks :sample, after slt, 1d
    Verify snapshot and record evidence :gate, after sample, 1d
    Compare compatible baseline :report, after gate, 1d

In [ ]:
timeline
    title Rollout dependencies in logical slots
    Agree on shared case IDs :ids, 2000-01-01, 1d
    Add manifest references to existing checks :reuse, after ids, 1d
    Add metric registry view to tracker history :metrics, after reuse, 1d
    Exercise matched comparison and failure fixtures :gate, after metrics, 1d
    Expand case and provider coverage :expand, after gate, 1d

## Jev, Z3 and proof scope

| Formal check | Jev interpretation | Z3 result and negative control |
|---|---|---|
| Declared metric bindings match allowed stage, unit, capture and grain tuples | Open, `jev-1.14.0`, weakest decision data completeness 0.99. Verify `data_integrity.mutually_consistent` on `metric_contract`. | All 12 rows pass in batches of 5, 5 and 2. Receipts `sol_598bcc7133334509`, `sol_94a169e15e2f4805`, `sol_23d7ebc763b14272`. Reclassifying native engine time as release capture fails, `sol_f1eae4a1e48448d0`. |
| Every complete receipt key references an existing case revision and scale | Open, `jev-1.14.0`, weakest decision data completeness 0.98. Verify `data_integrity.foreign_key` on `receipt_workload`. | Valid projection passes, `sol_16559269899b4522`. Pointing the ordered-limit receipt at a missing scaled instance fails, `sol_fe543f314a214748`. |

The initial metric request exceeded the catalog compiler's 64-variable limit. The execution requests partitioned the active rows and kept the exact rule, subject, schema and allowed tuples. The negative controls each change one fact. The foreign-key example assumes complete keys. Require key presence in the actual manifest and receipt validation because the catalog binding uses MATCH SIMPLE.

The diagram solver verifies each declared model against its profile. This covers stage topology, proof obligations for explicit dispatch/publication decisions, bounded message order, local state invariants, dependency feasibility, scan-count conservation and numeric consistency. SQLLogic and benchmark execution provide compiled-runtime evidence. The source notebooks and tracker snapshots identify the fixed chart inputs.

## NS-Mermaid coverage and visual appendix

All 13 executable headers in the current registry are used for the formal design. The remaining 16 registered headers have visual examples below. Their Markdown renders in Mermaid versions that support the listed syntax. The registry uses some short names where the Mermaid renderer uses a beta keyword. For profiles with no Mermaid header, the table records their current status.

| Registry profile | Headers or scope | Evidence in this notebook |
|---|---|---|
| Architecture policy | `architecture`, `architecture-beta` | Benchmark evidence and native correctness topologies |
| Relational LIA | `flowchart`, `graph` | Snapshot acceptance and comparison classification |
| State invariant LIA | `stateDiagram`, `stateDiagram-v2` | Generation diagnostics and verified sample retention |
| Sequence trace | `sequenceDiagram` | Release sample and separate diagnostic ordering |
| Schedule feasibility | `gantt`, `timeline` | Verification and rollout dependencies in logical slots |
| Flow conservation | `sankey` | Scan-row count lineage per arm |
| Numeric series | `pie`, `xychart`, `xychart-beta` | Verified-call counts, release p50 and scan rows |
| Numeric series with unavailable adapters | `radar`, `quadrant`, `treemap` | Visual coverage targets, investigation priorities and sample allocation |
| Unavailable formal adapters | `kanban`, `venn`, `packet`, `packet-beta`, `classDiagram`, `erDiagram`, `requirementDiagram`, `gitGraph` | Visual work queue, eligibility sets, digest fields, data model, requirements and rollout ancestry |
| Visualization-only concepts | `mindmap`, `journey`, `ishikawa`, `wardley`, `cynefin` | Navigation, replay workflow, drift causes, reuse posture and triage |
| Unavailable profiles without a header | Bounded reachability, transition BMC, packet encoding, event trace, schedule optimization | Registry status. The state and sequence views prove only their declared finite local obligations. |


#### classDiagram · Contracts resolved by the existing runners

```mermaid
classDiagram
    class WorkloadCase {
        String case_id
        Integer revision
        String sql_sha256
        String oracle_id
    }
    class WorkloadInstance {
        String fixture_id
        String parameter_trace_id
        String scale
    }
    class SqlLogicReceipt {
        String source_sha256
        Integer record_count
        String status
    }
    class TrackerRun {
        String run_id
        String artifact_sha256
        Integer protocol
    }
    class MetricObservation {
        String metric_id
        String capture
        String unit
        String sample_grain
    }
    WorkloadCase "1" --> "*" WorkloadInstance
    WorkloadInstance "1" --> "*" SqlLogicReceipt
    WorkloadInstance "*" --> "*" TrackerRun : measured in
    TrackerRun "1" --> "*" MetricObservation
```

#### erDiagram · History join path

```mermaid
erDiagram
    CASE_REVISION ||--o{ WORKLOAD_INSTANCE : defines
    WORKLOAD_INSTANCE ||--o{ SQLLOGIC_RECEIPT : verifies
    WORKLOAD_INSTANCE ||--o{ WORKER_SERIES : measures
    TRACKER_RUN ||--o{ WORKER_SERIES : contains
    WORKER_SERIES ||--o{ OBSERVATION : records
    METRIC_DEFINITION ||--o{ OBSERVATION : classifies
```

#### requirementDiagram · Acceptance at the two gates

```mermaid
requirementDiagram
    requirement exact_result {
        id: 1
        text: "Preserve typed SQL result and multiplicity"
        risk: high
        verifymethod: test
    }
    requirement matched_arms {
        id: 2
        text: "Compare verified arms from one workload cohort"
        risk: high
        verifymethod: test
    }
    element sql_logic {
        type: SQLLogic
    }
    element pair_check {
        type: Tracker
    }
    sql_logic - verifies -> exact_result
    pair_check - verifies -> matched_arms
```

Both packet-style views illustrate the logical evidence fields. The 256-bit widths represent SHA-256 digests. The JSON tracker has named fields and a versioned schema.

#### packet · Workload evidence digest fields

```mermaid
packet
    0-255: "Logical fixture SHA-256"
    256-511: "Parameter trace SHA-256"
    512-767: "Oracle result SHA-256"
```

#### packet-beta · Execution evidence digest fields

```mermaid
packet-beta
    0-255: "Extension SHA-256"
    256-511: "DuckDB host SHA-256"
    512-767: "Tracker code SHA-256"
```

#### venn · Shared case and cohort eligibility

```mermaid
venn-beta
    title Cases eligible for comparison
    set Correct["Oracle verified"]
    set Compatible["Identical workload cohort"]
    set Paired["Matched runner blocks"]
    union Correct,Compatible,Paired["Comparable series"]
```

#### kanban · Implementation queue

```mermaid
kanban
    ready[Ready]
        ids[Add versioned case records]
        witnesses[Link current SQLLogic cases]
    follow[After case records]
        metricview[Add normalized history observations]
        pair[Extend cohort pairing checks]
    check[Acceptance]
        smoke[Replay both scales and run negative controls]
```

#### gitGraph · Proposed change order

```mermaid
gitGraph
    commit id: "Existing SLT and tracker"
    branch workload
    checkout workload
    commit id: "Manifest references"
    commit id: "Tiny shared fixture"
    checkout main
    merge workload
    commit id: "Metrics and gates"
```

[Mermaid packet syntax](https://mermaid.js.org/syntax/packet.html) · [Venn syntax](https://mermaid.js.org/syntax/venn.html)

#### radar · Planned coverage targets

```mermaid
radar-beta
    title Coverage targets (one means required)
    axis schema["Schema"], bag["Bag semantics"], route["Route probe"], timing["Latency"], pairing["Pair checks"]
    curve pr["PR correctness"]{1, 1, 1, 0, 0}
    curve benchmark["Measured comparison"]{1, 1, 1, 1, 1}
    min 0
    max 1
```

The next coordinates show proposed investigation priorities on a relative scale.

#### quadrant · Which diagnostics to inspect first

```mermaid
quadrantChart
    title Investigation priorities
    x-axis Lower rerun cost --> Higher rerun cost
    y-axis Lower stage detail --> Higher stage detail
    quadrant-1 Deep profiling
    quadrant-2 Targeted evidence
    quadrant-3 Quick gates
    quadrant-4 Broad replay
    Oracle digest: [0.10, 0.20]
    Applied route log: [0.30, 0.75]
    Matched latency replay: [0.75, 0.45]
    Provider phase trace: [0.80, 0.85]
```

#### treemap · Measured calls in one local replay

```mermaid
treemap-beta
    "Ladybug"
        "Optimized": 4
        "Host": 4
    "RelateDB"
        "Optimized": 4
        "Host": 4
```

[Mermaid radar syntax](https://mermaid.js.org/syntax/radar.html) · [Quadrant syntax](https://mermaid.js.org/syntax/quadrantChart.html) · [Treemap syntax](https://mermaid.js.org/syntax/treemap.html)

#### mindmap · Evidence entry points

```mermaid
mindmap
  root((Investigate a hybrid case))
    Correctness
      Workload instance
      SQLLogic receipt
      Oracle result
    Execution
      Applied route
      Admitted generation
      Delivered graph rows
    Performance
      Verified warm samples
      Paired candidate and baseline
      Snapshot history
```

The journey scores are diagram inputs used to depict the workflow.

#### journey · How a contributor reproduces a series

```mermaid
journey
    title Reproduce one benchmark series
    section Find the case
      Open history attempt: 3: Contributor
      Resolve case revision and fixture: 3: Contributor
    section Replay
      Run referenced SQLLogic: 3: Contributor
      Run existing tracker: 3: Contributor
    section Explain the change
      Compare row counts and equal cohorts: 3: Contributor, Reviewer
```

#### ishikawa · Causes to check for replay drift

```mermaid
ishikawa-beta
    Replay drift
    Workload
        Changed SQL template
        Different typed parameter order
    Fixture
        Changed logical edges
        Missing provider materialization
    Observation
        Partial Arrow fetch
        Diagnostic build measured
    Runner
        Changed thread mode
        Unmatched case order
```

The Wardley coordinates represent the proposed reuse posture.

#### wardley · Reuse mature runners and add case lineage

```mermaid
wardley-beta
    title Test and benchmark reuse
    component Reproducible hybrid evidence [0.90, 0.73]
    component Workload manifest [0.72, 0.45]
    component SQLLogic runner [0.57, 0.86]
    component Matched provider tracker [0.55, 0.78]
    component DuckDB host artifact [0.28, 0.89]
    Reproducible hybrid evidence -> Workload manifest
    Workload manifest -> SQLLogic runner
    Workload manifest -> Matched provider tracker
    Matched provider tracker -> DuckDB host artifact
```

#### cynefin · Route triage by known evidence

```mermaid
cynefin-beta
    title Triage benchmark changes

    clear
    "Wrong oracle digest"

    complicated
    "Applied route changed"

    complex
    "Latency drift with identical scan rows"

    chaotic
    "Unreliable CI runner"

    confusion
    "Missing capture provenance"

    complex --> complicated : "Reproduce on same runner"
    chaotic --> complex : "Restore runner stability"
```

[Mermaid Ishikawa syntax](https://mermaid.js.org/syntax/ishikawa.html) · [Wardley syntax](https://mermaid.js.org/syntax/wardley.html) · [Cynefin syntax](https://mermaid.js.org/syntax/cynefin.html)